In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.varmax import VARMAX
from statsmodels.tsa.vector_ar.var_model import VAR
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, root_mean_squared_error

# 1. Veri Setini Yükleme
file_path = "data/selected_lip_coordinates.csv"
df = pd.read_csv(file_path)

# 2. Eksik Veri Kontrolü
print("Missing values:")
print(df.isnull().sum())

# Zaman bilgisini x ekseni olarak belirle
#time = df.index

# 3. Zaman Serisi Görselleştirme
#plt.figure(figsize=(12, 6))

# Her sütunu ayrı ayrı çiz
#for col in df.columns:
    #plt.plot(time, df[col], label=col)

# Grafik ayarları
#plt.xlabel("Time")
#plt.ylabel("Value")
#plt.title("Time Series Plot of Multiple Features")
#plt.legend(title="Features")
#plt.grid(True)
#plt.show()
#plt.close()

Missing values:
time    0
0_x     0
13_x    0
14_x    0
17_x    0
37_x    0
0_y     0
13_y    0
14_y    0
17_y    0
37_y    0
dtype: int64


In [24]:
# 4. ADF Durağanlık Testi ve Durağan Olmayanlara Fark Alma
#def adf_test(series):
    #result = adfuller(series.dropna())
    #print(f"ADF Statistic: {result[0]}")
    #print(f"p-value: {result[1]}")
    #print("Critical Values:")
    #for key, value in result[4].items():
        #print(f"   {key}: {value}")
    #if result[1] <= 0.05:
        #print("Seri durağandır.")
        #return True
    #else:
        #print("Seri durağan değildir.")
        #return False

#print("Durağanlık Testi Sonuçları:")
#non_stationary_columns = []
#for column in df.columns:
    #print(f"\n{column} için ADF Testi:")
    #is_stationary = adf_test(df[column])
    #if not is_stationary:
        #non_stationary_columns.append(column)

#print(f"Durağan olmayan sütunlar: {non_stationary_columns}")

In [25]:
# 5. ACF ve PACF Grafikleri
##fig, ax = plt.subplots(2, 1, figsize=(12, 8))
#plot_acf(df.dropna().iloc[:, 0], ax=ax[0], lags=20)
#ax[0].set_title("Autocorrelation Function (ACF)")
#plot_pacf(df.dropna().iloc[:, 0], ax=ax[1], lags=20)
#ax[1].set_title("Partial Autocorrelation Function (PACF)")
#plt.subplots_adjust(hspace=0.7, wspace=0.4)
#plt.show()

In [42]:
# Veri Setini Train ve Test olarak Bölme
# 'time' sütununu çıkar ve yalnızca koordinatları modele dahil et
data = df.drop(columns=['time'])
train_size = int(len(data) * 0.8)  # %80 eğitim, %20 test
train, test = data[0:train_size], data[train_size:len(data)]

# Fit VAR model
model = VAR(train)
model_fit = model.fit(maxlags=15, ic='aic')

# Model özeti
print(model_fit.summary())

# Test seti üzerinde tahmin yapma
lag_order = model_fit.k_ar
forecast_input = train.values[-lag_order:]
forecast = model_fit.forecast(y=forecast_input, steps=len(test))

# Tahminleri DataFrame'e dönüştürme
predictions = pd.DataFrame(forecast, index=test.index, columns=test.columns)


# Performans Metriklerini Hesaplama

mae = mean_absolute_error(test, predictions)
mse = mean_squared_error(test, predictions)
rmse = root_mean_squared_error(test, predictions)
mape = mean_absolute_percentage_error(test, predictions)

print("Gerçek Değerler vs Tahmin Edilen Değerler:")
for expected, predicted in zip(test.values, predictions.values):
    print(f"Gerçek: {expected}, \n\nTahmin: {predicted}")
    print("------------------------------------------------------------------------------")

print("------------------------------------------------------------------------------")
print(f"Mean Absolute Error (MAE): {mae}")
print(f"Mean Squared Error (MSE): {mse}")
print(f"Root Mean Squared Error (RMSE): {rmse}")
print(f"Mean Absolute Percentage Error (MAPE): {mape}%")
# plot forecasts against actual outcomes
# plt.plot(test)
# plt.plot(predictions, color='red')
# plt.show()

  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Tue, 25, Mar, 2025
Time:                     21:27:23
--------------------------------------------------------------------
No. of Equations:         10.0000    BIC:                   -10.8434
Nobs:                     876.000    HQIC:                  -11.8870
Log likelihood:          -6630.29    FPE:                3.60545e-06
AIC:                     -12.5334    Det(Omega_mle):     2.54640e-06
--------------------------------------------------------------------
Results for equation 0_x
             coefficient       std. error           t-stat            prob
--------------------------------------------------------------------------
const          15.929734         6.137886            2.595           0.009
L1.0_x          0.450516         0.082330            5.472           0.000
L1.13_x         0.425815         0.071211            5.980           0.000
L1.14_x    

In [43]:
# Veri Setini Train ve Test olarak Bölme
# 'time' sütununu çıkar ve yalnızca koordinatları modele dahil et
data = df.drop(columns=['time'])
train_size = int(len(data) * 0.8)  # %80 eğitim, %20 test
train, test = data[0:train_size], data[train_size:len(data)]

history = train.values.tolist()
predictions = []

# Walk-forward validation ile tahmin etme
for t in range(len(test)):
    model = VAR(pd.DataFrame(history, columns=data.columns))
    model_fit = model.fit(maxlags=15, ic='aic')
    lag_order = model_fit.k_ar
    forecast_input = np.array(history[-lag_order:])
    forecast = model_fit.forecast(y=forecast_input, steps=1)
    predictions.append(forecast[0])
    history.append(test.values[t])

# Tahminleri DataFrame'e dönüştürme
#predictions_df = pd.DataFrame(predictions, index=test.index, columns=test.columns)


# Performans Metriklerini Hesaplama

mae = mean_absolute_error(test, predictions)
mse = mean_squared_error(test, predictions)
rmse = root_mean_squared_error(test, predictions)
mape = mean_absolute_percentage_error(test, predictions)

print("Gerçek Değerler vs Tahmin Edilen Değerler:")
for expected, predicted in zip(test.values, predictions):
    print(f"Gerçek: {expected}, \n\nTahmin: {predicted}")
    print("------------------------------------------------------------------------------")

print("------------------------------------------------------------------------------")
print(f"Mean Absolute Error (MAE): {mae}")
print(f"Mean Squared Error (MSE): {mse}")
print(f"Root Mean Squared Error (RMSE): {rmse}")
print(f"Mean Absolute Percentage Error (MAPE): {mape}%")
# plot forecasts against actual outcomes
# # plt.plot(test)
# plt.plot(predictions, color='red')
# plt.show()

Gerçek Değerler vs Tahmin Edilen Değerler:
Gerçek: [631 632 632 633 621 326 337 337 352 325], 

Tahmin: [631.19066181 631.66364907 632.35213257 633.07729771 621.559524
 326.14192219 335.84417548 337.15909177 351.33855829 324.93292016]
------------------------------------------------------------------------------
Gerçek: [630 630 631 632 620 326 337 337 351 325], 

Tahmin: [630.40315513 630.99115574 631.59014049 632.3358128  620.63980072
 326.95686115 337.07428606 338.81826895 353.57468982 325.6826248 ]
------------------------------------------------------------------------------
Gerçek: [628 629 629 630 618 325 335 337 351 324], 

Tahmin: [628.79566582 629.3672378  629.97036039 630.70700495 619.13072067
 325.76394468 336.17947771 335.88933245 350.14290205 324.53126796]
------------------------------------------------------------------------------
Gerçek: [627 628 628 629 618 323 333 337 352 322], 

Tahmin: [626.46597643 627.1929766  627.69006037 628.36492043 616.60767897
 323.78081996